# Wan2GP в Google Colab (русская версия)

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GamidKambulatov05/wan2gp-colab-ru/blob/main/wan2gp-colab-ru.ipynb)

Настройка [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) в новой GPU-сессии Google Colab.

Запускай ячейки по порядку — это подготовит окружение, установит зависимости и запустит веб-интерфейс Gradio. В выводе последней ячейки появится ссылка — перейди по ней, чтобы открыть интерфейс в браузере.

> **О видеопамяти в Colab:** бесплатный тариф обычно выдаёт GPU T4 с 15 ГБ. Большинству моделей Wan2GP этого не хватает — но модель Wan 2.2 TextImage2Video FastWan работает и на T4, генерируя 5-секундный клип в 480p примерно за 8 минут. На платной подписке Colab Pro (A100, 40-80 ГБ) доступны полноразмерные модели в 720p.

> **Совет:** перед первой генерацией снизь разрешение в интерфейсе Wan2GP. По умолчанию стоит 1280x720 — бесплатному GPU в Colab это может быть не по силам.

> Ноутбук основан на [Square-Zero-Labs/Wan2GP-on-Colab](https://github.com/Square-Zero-Labs/Wan2GP-on-Colab) (лицензия Apache-2.0), переведён и адаптирован на русский язык.

## Шаг 1. Проверяем видеокарту (GPU)

Перед запуском зайди в `Runtime → Change runtime type` и выбери **GPU** (в идеале A100, если он у тебя доступен).

Если эта ячейка выдаст ошибку — значит GPU не подключён. Вернись в `Runtime → Change runtime type`, выбери **GPU**, сохрани и запусти ячейку заново.

In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU не найден. В Colab открой Runtime → Change runtime type, выбери GPU, сохрани и запусти эту ячейку заново.'
    ) from exc

## Шаг 2. Настраиваем пути и (опционально) постоянное хранилище на Google Drive

Поставь `USE_GOOGLE_DRIVE_DATA = True`, если хочешь, чтобы чекпоинты, LoRA, результаты и кэш моделей сохранялись между сессиями Colab через Google Drive. Иначе всё будет храниться на временном диске Colab и удалится при завершении сессии.

In [ ]:
from pathlib import Path


# ПОМЕНЯЙ НА True, ЕСЛИ ХОЧЕШЬ ХРАНИТЬ ДАННЫЕ НА GOOGLE DRIVE (СОХРАНЯЮТСЯ МЕЖДУ СЕССИЯМИ)
USE_GOOGLE_DRIVE_DATA = False


DRIVE_MOUNT_POINT = Path('/content/drive')
WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
EPHEMERAL_DATA_ROOT = Path('/content/Wan2GP-data').resolve()
PERSISTENT_DATA_ROOT = (DRIVE_MOUNT_POINT / 'MyDrive' / 'Wan2GP-data').resolve()

if USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    WAN_DATA_ROOT = PERSISTENT_DATA_ROOT
    data_mode = 'Google Drive (постоянное хранилище)'
else:
    WAN_DATA_ROOT = EPHEMERAL_DATA_ROOT
    data_mode = 'диск сессии Colab (временное хранилище)'

WAN_CKPTS_DIR = (WAN_DATA_ROOT / 'ckpts').resolve()
WAN_LORAS_DIR = (WAN_DATA_ROOT / 'loras').resolve()
WAN_OUTPUTS_DIR = (WAN_DATA_ROOT / 'outputs').resolve()
WAN_CACHE_DIR = (WAN_DATA_ROOT / 'cache').resolve()
WAN_LTX2_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2').resolve()
WAN_LTX2_22B_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2_22B').resolve()

WAN2GP_ROOT.parent.mkdir(parents=True, exist_ok=True)
WAN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
WAN_CKPTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_22B_LORAS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Путь к репозиторию Wan2GP: {WAN2GP_ROOT}')
print(f'Режим хранения данных: {data_mode}')
print(f'Корневая папка данных: {WAN_DATA_ROOT}')
print(f'Чекпоинты: {WAN_CKPTS_DIR}')
print(f'LoRA: {WAN_LORAS_DIR}')
print(f'LTX-2 LoRA: {WAN_LTX2_LORAS_DIR}')
print(f'LTX-2 22B LoRA: {WAN_LTX2_22B_LORAS_DIR}')
print(f'Результаты: {WAN_OUTPUTS_DIR}')
print(f'Кэш: {WAN_CACHE_DIR}')

## Шаг 3. Скачиваем или обновляем Wan2GP

Клонируем репозиторий, если его ещё нет, либо подтягиваем последние изменения, если он уже был склонирован ранее.

In [ ]:
import shutil, subprocess
from pathlib import Path

def merge_directory_contents(source_dir: Path, destination_dir: Path) -> None:
    for child in list(source_dir.iterdir()):
        destination = destination_dir / child.name
        if destination.exists():
            if child.is_dir() and destination.is_dir():
                merge_directory_contents(child, destination)
                child.rmdir()
                continue
            raise RuntimeError(f'Не могу переместить {child} в {destination_dir}: {destination} уже существует.')
        shutil.move(str(child), str(destination))

def attach_data_directory(repo_path: Path, data_path: Path) -> None:
    data_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        if repo_path.resolve() != data_path.resolve():
            raise RuntimeError(f'{repo_path} уже указывает на {repo_path.resolve()}, а ожидалось {data_path}.')
        print(f'Используем существующую связь: {repo_path} -> {data_path}')
        return

    if repo_path.exists():
        if not repo_path.is_dir():
            raise RuntimeError(f'Ожидалась папка по пути {repo_path}.')
        merge_directory_contents(repo_path, data_path)
        repo_path.rmdir()
    else:
        repo_path.parent.mkdir(parents=True, exist_ok=True)

    repo_path.symlink_to(data_path, target_is_directory=True)
    print(f'Связали {repo_path.name} -> {data_path}')

MANAGED_PATHS = {'ckpts', 'loras', 'outputs', 'ffmpeg_bins'}

repo_url = 'https://github.com/deepbeepmeep/Wan2GP.git'
if WAN2GP_ROOT.exists():
    untracked = subprocess.run(
        ['git', '-C', str(WAN2GP_ROOT), 'ls-files', '--others', '--exclude-standard'],
        check=True,
        capture_output=True,
        text=True,
    )
    user_files = [
        line for line in untracked.stdout.splitlines()
        if line.split('/', 1)[0] not in MANAGED_PATHS
    ]
    if user_files:
        preview = ', '.join(user_files[:5]) + (' …' if len(user_files) > 5 else '')
        print(f'В репозитории есть твои собственные файлы ({preview}). Пропускаем обновление, чтобы их не потерять.')
    else:
        print('Репозиторий уже склонирован. Обновляем до последней версии...')
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', '--', '.'], check=True)
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

attach_data_directory(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
attach_data_directory(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
attach_data_directory(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)

## Шаг 4. Ставим системные зависимости

Устанавливаем общие библиотеки для обработки видео и звука, а также совместимую сборку FFmpeg, если встроенная в Colab версия слишком старая. Загрузка FFmpeg проверяется по контрольной сумме (SHA-256) отдельно от системных пакетов. Именно этот шаг раньше занимал у нас часы из-за сборки flash-attn из исходников — здесь такого нет, всё ставится готовыми файлами. Если увидишь предупреждение о пропуске какого-то дополнительного репозитория — это не страшно, можно игнорировать.

In [ ]:
import hashlib, json, os, platform, re, shutil, subprocess, tarfile, tempfile, urllib.request
from pathlib import Path

APT_PACKAGES = ['libglib2.0-0', 'libgl1', 'libportaudio2']
FFMPEG_RELEASE_API = 'https://api.github.com/repos/BtbN/FFmpeg-Builds/releases/latest'
FFMPEG_DOWNLOAD_PREFIX = 'https://github.com/BtbN/FFmpeg-Builds/releases/download/'
FFMPEG_BIN_DIR = WAN2GP_ROOT / 'ffmpeg_bins'
FFMPEG_CACHE_DIR = WAN_CACHE_DIR / 'downloads'
FFMPEG_ARCHITECTURES = {
    'x86_64': 'linux64',
    'amd64': 'linux64',
    'aarch64': 'linuxarm64',
    'arm64': 'linuxarm64',
}

def dpkg_installed(package: str) -> bool:
    return subprocess.run(['dpkg', '-s', package], capture_output=True).returncode == 0

missing = [package for package in APT_PACKAGES if not dpkg_installed(package)]
if missing:
    env = os.environ.copy()
    env['DEBIAN_FRONTEND'] = 'noninteractive'
    print('Устанавливаем:', ', '.join(missing))
    subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
    subprocess.run([
        'sudo', 'apt-get', 'install', '-y', '--no-install-recommends', *missing
    ], check=True, env=env)
else:
    print('Системные библиотеки уже установлены.')

def ffmpeg_supports_required_options(binary: Path) -> bool:
    try:
        result = subprocess.run(
            [str(binary), '-hide_banner', '-h', 'full'],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=30,
        )
    except (OSError, subprocess.TimeoutExpired):
        return False
    option_pattern = re.compile(r'^\s*-fps_mode(?:\[:[^]]+\])?(?:\s|$)', re.MULTILINE)
    return result.returncode == 0 and option_pattern.search(result.stdout) is not None

def executable_works(binary: Path) -> bool:
    try:
        return subprocess.run(
            [str(binary), '-version'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=30,
        ).returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        return False

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def github_json(url: str) -> dict:
    request = urllib.request.Request(
        url,
        headers={
            'Accept': 'application/vnd.github+json',
            'User-Agent': 'Wan2GP-Colab-notebook-ru',
            'X-GitHub-Api-Version': '2022-11-28',
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)

def download_file(url: str, destination: Path, expected_size: int) -> None:
    request = urllib.request.Request(url, headers={'User-Agent': 'Wan2GP-Colab-notebook-ru'})
    with urllib.request.urlopen(request, timeout=120) as response, destination.open('wb') as output:
        downloaded = 0
        next_report = 10
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
            downloaded += len(chunk)
            if expected_size:
                percent = downloaded * 100 // expected_size
                if percent >= next_report:
                    print(f'Загрузка FFmpeg: {min(percent, 100)}%')
                    next_report += 10
    if expected_size and downloaded != expected_size:
        raise RuntimeError(f'Загрузка FFmpeg не завершена: ожидалось {expected_size} байт, получено {downloaded}.')

def install_current_stable_ffmpeg() -> tuple[Path, Path]:
    machine = platform.machine().lower()
    asset_architecture = FFMPEG_ARCHITECTURES.get(machine)
    if asset_architecture is None:
        raise RuntimeError(f'Для архитектуры {machine} нет подходящей сборки FFmpeg.')
    asset_pattern = re.compile(
        rf'^ffmpeg-n(?P<version>\d+\.\d+)-latest-{asset_architecture}-gpl-(?P=version)\.tar\.xz$'
    )

    release = github_json(FFMPEG_RELEASE_API)
    candidates = []
    for asset in release.get('assets', []):
        match = asset_pattern.fullmatch(asset.get('name', ''))
        if match:
            version = tuple(int(part) for part in match.group('version').split('.'))
            candidates.append((version, asset))
    if not candidates:
        raise RuntimeError(f'Стабильная сборка FFmpeg для {asset_architecture} не найдена в релизе.')

    _, asset = max(candidates, key=lambda item: item[0])
    asset_url = asset.get('browser_download_url', '')
    digest = asset.get('digest') or ''
    if not asset_url.startswith(FFMPEG_DOWNLOAD_PREFIX):
        raise RuntimeError(f'Неожиданная ссылка на загрузку FFmpeg: {asset_url}')
    if not digest.startswith('sha256:'):
        raise RuntimeError('В релизе FFmpeg нет SHA-256 контрольной суммы — отказываемся от непроверенной загрузки.')

    expected_sha256 = digest.split(':', 1)[1].lower()
    expected_size = int(asset.get('size') or 0)
    FFMPEG_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = FFMPEG_CACHE_DIR / asset['name']
    archive_valid = (
        archive_path.is_file()
        and (not expected_size or archive_path.stat().st_size == expected_size)
        and sha256_file(archive_path) == expected_sha256
    )
    if archive_valid:
        print(f'Используем проверенный кэш FFmpeg: {archive_path.name}')
    else:
        print(f'Скачиваем стабильную сборку FFmpeg: {asset["name"]}')
        with tempfile.NamedTemporaryFile(dir=FFMPEG_CACHE_DIR, suffix='.download', delete=False) as temp_file:
            temp_path = Path(temp_file.name)
        try:
            download_file(asset_url, temp_path, expected_size)
            actual_sha256 = sha256_file(temp_path)
            if actual_sha256 != expected_sha256:
                raise RuntimeError(
                    f'Контрольная сумма FFmpeg не совпадает: ожидалось {expected_sha256}, получено {actual_sha256}.'
                )
            os.replace(temp_path, archive_path)
        finally:
            temp_path.unlink(missing_ok=True)

    FFMPEG_BIN_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, mode='r:xz') as archive:
        members = {}
        for member in archive.getmembers():
            member_path = Path(member.name)
            if member.isfile() and member_path.parent.name == 'bin' and member_path.name in {'ffmpeg', 'ffprobe'}:
                if member_path.name in members:
                    raise RuntimeError(f'Дублирующийся файл {member_path.name} в архиве FFmpeg.')
                members[member_path.name] = member
        if set(members) != {'ffmpeg', 'ffprobe'}:
            raise RuntimeError('В архиве FFmpeg нет одновременно ffmpeg и ffprobe.')
        for binary_name, member in members.items():
            source = archive.extractfile(member)
            if source is None:
                raise RuntimeError(f'Не удалось прочитать {binary_name} из архива FFmpeg.')
            destination = FFMPEG_BIN_DIR / binary_name
            temporary_destination = FFMPEG_BIN_DIR / f'.{binary_name}.tmp'
            with source, temporary_destination.open('wb') as output:
                shutil.copyfileobj(source, output)
            temporary_destination.chmod(0o755)
            os.replace(temporary_destination, destination)

    return FFMPEG_BIN_DIR / 'ffmpeg', FFMPEG_BIN_DIR / 'ffprobe'

local_ffmpeg = FFMPEG_BIN_DIR / 'ffmpeg'
local_ffprobe = FFMPEG_BIN_DIR / 'ffprobe'
system_ffmpeg_name = shutil.which('ffmpeg')
system_ffprobe_name = shutil.which('ffprobe')
system_ffmpeg = Path(system_ffmpeg_name) if system_ffmpeg_name else None
system_ffprobe = Path(system_ffprobe_name) if system_ffprobe_name else None

if local_ffmpeg.exists() or local_ffprobe.exists():
    if ffmpeg_supports_required_options(local_ffmpeg) and executable_works(local_ffprobe):
        ffmpeg_binary, ffprobe_binary = local_ffmpeg, local_ffprobe
        print('Используем уже установленную совместимую сборку FFmpeg для Wan2GP.')
    else:
        print('Сборка FFmpeg для Wan2GP неполная или устаревшая, обновляем...')
        ffmpeg_binary, ffprobe_binary = install_current_stable_ffmpeg()
elif (
    system_ffmpeg is not None
    and system_ffprobe is not None
    and ffmpeg_supports_required_options(system_ffmpeg)
    and executable_works(system_ffprobe)
):
    ffmpeg_binary, ffprobe_binary = system_ffmpeg, system_ffprobe
    print('Используем совместимый системный FFmpeg из Colab.')
else:
    print('FFmpeg в Colab отсутствует или не поддерживает -fps_mode, ставим актуальную стабильную сборку...')
    ffmpeg_binary, ffprobe_binary = install_current_stable_ffmpeg()

if not ffmpeg_supports_required_options(ffmpeg_binary) or not executable_works(ffprobe_binary):
    raise RuntimeError('Проверка FFmpeg после установки не пройдена.')

os.environ['FFMPEG_BINARY'] = str(ffmpeg_binary.resolve())
os.environ['FFPROBE_BINARY'] = str(ffprobe_binary.resolve())
binary_dir = str(ffmpeg_binary.resolve().parent)
path_parts = [part for part in os.environ.get('PATH', '').split(os.pathsep) if part != binary_dir]
os.environ['PATH'] = os.pathsep.join([binary_dir, *path_parts])
version_line = subprocess.run(
    [os.environ['FFMPEG_BINARY'], '-version'], check=True, capture_output=True, text=True
).stdout.splitlines()[0]
print(f'FFmpeg готов: {version_line}')
print(f'Файл FFmpeg: {os.environ["FFMPEG_BINARY"]}')

## Шаг 5. Ставим Python-зависимости

Устанавливаем PyTorch и Python-пакеты Wan2GP. Займёт несколько минут — используем `uv` вместо обычного pip, он в разы быстрее. Если в Colab уже стоит подходящий PyTorch с поддержкой CUDA, переустанавливать его не будем — сэкономим время.

In [ ]:
import json, os, subprocess, sys

PYTORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu128'
REPLACEMENT_TORCH = ('torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0')
UNSUPPORTED_TORCH = ('2.8.0', '2.9.')
ONNXRUNTIME_GPU = 'onnxruntime-gpu==1.22.0'

TORCH_PROBE = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import importlib.metadata as md, torch\n'
    '    for name in ("torch", "torchvision", "torchaudio"):\n'
    '        try:\n'
    '            info[name] = md.version(name)\n'
    '        except Exception:\n'
    '            pass\n'
    '    info["cuda"] = torch.cuda.is_available()\n'
    '    if info["cuda"]:\n'
    '        info["device"] = torch.cuda.get_device_name(0)\n'
    'except Exception:\n'
    '    pass\n'
    'print(json.dumps(info))\n'
)

ONNX_PROBE = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import onnxruntime\n'
    '    info["version"] = onnxruntime.__version__\n'
    '    info["providers"] = onnxruntime.get_available_providers()\n'
    'except Exception:\n'
    '    pass\n'
    'print(json.dumps(info))\n'
)

env = os.environ.copy()

if USE_GOOGLE_DRIVE_DATA:
    uv_cache_dir = WAN_CACHE_DIR / 'uv'
    uv_cache_dir.mkdir(parents=True, exist_ok=True)
    env['UV_CACHE_DIR'] = str(uv_cache_dir)
    env['UV_LINK_MODE'] = 'copy'

def uv_pip(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', *args], check=True, env=env)

def uv_install(*args):
    uv_pip('install', '--system', *args)

def probe(code):
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, env=env)
    try:
        return json.loads(result.stdout.strip().splitlines()[-1])
    except Exception:
        return {}

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True, env=env)

torch_version = probe(TORCH_PROBE).get('torch', '')
try:
    torch_release = tuple(int(part) for part in torch_version.split('.')[:2])
except ValueError:
    torch_release = ()
torch_usable = (
    torch_release >= (2, 7)
    and not torch_version.startswith(UNSUPPORTED_TORCH)
)

if torch_usable:
    print(f'Используем PyTorch, уже установленный в этой сессии ({torch_version}).')
else:
    print('Устанавливаем PyTorch...')
    uv_install(*REPLACEMENT_TORCH, '--index-url', PYTORCH_INDEX_URL)

print('Устанавливаем пакеты Wan2GP...')
uv_install('-r', str(WAN2GP_ROOT / 'requirements.txt'), '--index-strategy', 'unsafe-best-match')

uv_install(ONNXRUNTIME_GPU)
onnx = probe(ONNX_PROBE)
if not onnx.get('version'):
    uv_pip('uninstall', '--system', 'onnxruntime-gpu')
    uv_install('onnxruntime>=1.22.0')
    onnx = probe(ONNX_PROBE)

report = probe(TORCH_PROBE)
if not report.get('torch'):
    raise RuntimeError(
        'PyTorch не работает после установки. Открой Runtime -> Restart session и заново выполни Шаги 2-5.'
    )
if not report.get('cuda'):
    raise RuntimeError(
        'GPU не найден. Открой Runtime -> Change runtime type, выбери GPU и заново выполни Шаги 1 и 5.'
    )

print(f"Готово: PyTorch {report['torch']} на {report['device']}.")
if 'CUDAExecutionProvider' not in onnx.get('providers', []):
    print('Примечание: удаление фона будет работать через CPU, а не GPU.')

## Шаг 5б. Принудительно переключаем matplotlib в безголовый режим

Гарантируем, что инструменты предобработки Wan2GP используют безголовый backend Agg — иначе Шаг 6 может упасть с ошибкой в Colab.

In [ ]:
from pathlib import Path

# Заменяем backend TkAgg на безголовый Agg, если он используется.
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Пропускаем: {target} не найден.')
else:
    text = target.read_text()
    if replacement in text:
        print('Backend Agg уже установлен, изменения не нужны.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Заменили TkAgg на Agg в interact_tools.py.')
    else:
        print('Вызов backend не найден, изменения не внесены.')

## Шаг 5в. Русский интерфейс и лёгкий рестайлинг (опционально)

Добавляем поверх WanGP: перевод вкладок и основных кнопок на русский, подсказки у фильтра моделей (что означает "All/Movie/Image/Audio"), небольшую плашку-подсказку "как выбрать модель" и лёгкий визуальный рестайлинг в духе тёмных тем ChatGPT/Claude. Полностью переводить все технические параметры (их сотни на каждую модель) в рамках этого слоя не пытаемся — переведено то, что видно по умолчанию и решает главный вопрос "какую модель выбрать".

Это отдельный слой поверх `shared/gradio/ui_styles.css` и `shared/gradio/ui_scripts.js` — не трогает сам код WanGP, поэтому переживает `git pull` в Шаге 3 (просто перезапусти эту ячейку после обновления репозитория).

In [ ]:
from pathlib import Path

RU_CSS = r'''
/* === RU-UI START === */
/* Русская тема WanGP: лёгкий рестайлинг поверх штатной тёмной темы Gradio,
   через штатные CSS-переменные темы (безопасно для раскладки/логики). */

:root {
  --color-accent: #7c5cff;
  --color-accent-soft: #7c5cff22;
  --button-primary-background-fill: #7c5cff;
  --button-primary-background-fill-hover: #6a4bef;
  --button-primary-text-color: #ffffff;
  --radius-sm: 8px;
  --radius-md: 12px;
  --radius-lg: 16px;
  --body-text-size: 15px;
}

body.dark {
  --background-fill-primary: #0e0f14;
  --background-fill-secondary: #15171f;
  --border-color-primary: #2a2d3a;
  --block-background-fill: #171923;
}

.tab-container[role="tablist"] button {
  border-radius: var(--radius-md) var(--radius-md) 0 0;
  font-weight: 600;
  letter-spacing: 0.01em;
}

.tab-container[role="tablist"] button.selected {
  color: var(--color-accent);
  box-shadow: inset 0 -3px 0 var(--color-accent);
}

button.primary, button[variant="primary"] {
  border-radius: var(--radius-md);
  box-shadow: 0 2px 10px var(--color-accent-soft);
}

.ru-model-guide {
  margin: 8px 0 4px 0;
  padding: 10px 14px;
  border-radius: var(--radius-md);
  background: var(--color-accent-soft);
  border: 1px solid var(--color-accent);
  font-size: 13px;
  line-height: 1.45;
}

.ru-model-guide b {
  color: var(--color-accent);
}

.ru-workflow-guide {
  margin: 6px 0 10px 0;
  padding: 10px 14px;
  border-radius: var(--radius-md);
  background: #22c55e18;
  border: 1px solid #22c55e;
  font-size: 13px;
  line-height: 1.6;
}

.ru-workflow-guide b {
  color: #22c55e;
}

::-webkit-scrollbar {
  width: 10px;
  height: 10px;
}

::-webkit-scrollbar-thumb {
  background: var(--border-color-primary);
  border-radius: var(--radius-sm);
}
/* === RU-UI END === */
'''

RU_JS = r'''
/* === RU-UI START === */
(function () {
  "use strict";

  // Переводим по технической id вкладки — надёжно, не ломается при обновлениях WanGP.
  var TAB_LABELS = {
    media_gen: "Генератор медиа",
    plugin_mask_generator: "Генератор масок",
    plugin_motion_designer: "Дизайнер движения",
    plugin_info: "Гайды",
    plugin_configuration: "Настройки",
    plugin_plugin_manager_tab: "Плагины",
    plugin_about_tab: "О программе",
    t2v: "Текст в видео",
    t2i: "Текст в изображение",
  };

  // Точный перевод часто встречающихся кнопок/подписей.
  var TEXT_MAP = {
    "Apply": "Применить",
    "Refresh": "Обновить",
    "Save": "Сохранить",
    "Delete": "Удалить",
    "Generate": "Сгенерировать",
    "Run": "Запустить",
    "Cancel": "Отмена",
    "Confirm": "Подтвердить",
    "Confirm Delete": "Подтвердить удаление",
    "Create": "Создать",
    "Create & New": "Создать и новый",
    "Create New Finetune": "Создать новый файнтюн",
    "Don't do it !": "Отмена",
    "Go Ahead Save it !": "Да, сохранить!",
    "Go Ahead Delete it !": "Да, удалить!",
    "Silent Cancel": "Тихая отмена",
    "Clear": "Очистить",
    "Clear Queue": "Очистить очередь",
    "Add": "Добавить",
    "Add New Prompt To Queue": "Добавить промпт в очередь",
    "Remove": "Убрать",
    "Exit": "Выход",
    "Abort": "Прервать",
    "Stop": "Стоп",
    "Pause": "Пауза",
    "Resume": "Продолжить",
    "Apply Audio Postprocessing": "Применить постобработку звука",
    "Apply Edits": "Применить изменения",
    "Apply Postprocessing": "Применить постобработку",
    "Download Lora": "Скачать LoRA",
    "Enter Lora URL": "Введите ссылку на LoRA",
    "Export Settings to File": "Экспортировать настройки в файл",
    "Extend this Sample": "Продлить этот пример",
    "Extract Settings": "Извлечь настройки",
    "Extracting": "Извлечение…",
    "Import Finetune JSON": "Импортировать JSON файнтюна",
    "Load Settings From Media File / Json / Zip": "Загрузить настройки из медиафайла / JSON / ZIP",
    "One More Sample": "Ещё один пример",
    "Reset Settings": "Сбросить настройки",
    "Save and Quit": "Сохранить и выйти",
    "Set Settings as Default": "Сделать настройками по умолчанию",
    "Close information": "Закрыть информацию",
    "Eject Audio": "Извлечь аудио",
    "Eject Audio File": "Извлечь аудиофайл",
    "Eject Deleted File": "Извлечь удалённый файл",
    "Eject Image": "Извлечь изображение",
    "Eject Media": "Извлечь медиафайл",
    "Eject Video": "Извлечь видео",
    "Upload file": "Загрузить файл",
    "Capture from camera": "Снять камерой",
    "Record audio": "Записать аудио",
    "Paste from clipboard": "Вставить из буфера",
    "Prompt Guidelines": "Советы по промпту",
    "Dictate prompt": "Надиктовать промпт",
    "Prompt Helper": "Помощник промпта",
    "Enhance Prompt": "Улучшить промпт",
    "Right ▶": "Вправо ▶",
    "◀ Left": "◀ Влево",
    "Folder": "Папка",
    "Up": "Вверх",
    "Brush": "Кисть",
    "New Video": "Новое видео",
    "Text Prompt": "Только текст",
    "Start with Image": "Начать с изображения",
    "Continue Video": "Продолжить видео",
    "Continue Last Video": "Продолжить последнее видео",
    "Video / Images Gallery": "Галерея видео / изображений",
    "Audio Files Gallery": "Галерея аудио",
    "Audio Gallery": "Галерея аудио",
    "Video to Continue": "Видео для продолжения",
    "Generated videos": "Сгенерированные видео",
    "MULTI image": "Несколько изображений",
    "ONE image": "Одно изображение",
    "All model families": "Все семейства моделей",
    "Movie model families": "Видео-модели",
    "Image-only model families": "Только изображения",
    "Audio-only model families": "Только аудио",
    "Search models": "Поиск моделей",
    "Apply model search": "Применить поиск моделей",
    "Close model search": "Закрыть поиск моделей",
    "Apply model output filter": "Применить фильтр моделей",
    "Choose a Lora Preset or a Settings file in this List": "Выбери пресет LoRA или файл настроек из списка",
    "No matching LoRAs": "Подходящих LoRA не найдено",
    "Activated LoRAs": "Активные LoRA",
    "Always Loaded LoRAs": "Всегда загружаемые LoRA",
    "LoRAs Multipliers": "Множители LoRA",
    "Lora URL": "Ссылка на LoRA",
    "LoRAs Multipliers (1.0 by default) separated by Space chars or CR, lines that start with # are ignored": "Множители LoRA (по умолчанию 1.0), через пробел или новую строку; строки с # игнорируются",
    "model mode": "режим модели",
    "1024x2048": "1024x2048",
    ">=720&<=1440": ">=720&<=1440",
    "Custom Resolutions (one WxH value per line)": "Свои разрешения (по одному ШxВ на строку)",
    "Description": "Описание",
    "Finetune Parameter 1": "Параметр файнтюна 1",
    "Finetune Parameter 2": "Параметр файнтюна 2",
    "Finetune Parameter 3": "Параметр файнтюна 3",
    "Id": "ID",
    "Main Checkpoints": "Основные чекпоинты",
    "Max Tokens (empty = auto)": "Макс. токенов (пусто = автоматически)",
    "Name": "Название",
    "Resolution Categories Conditions (OR operator between lines)": "Условия категорий разрешения (условие ИЛИ между строками)",
    "Secondary Checkpoints": "Дополнительные чекпоинты",
    "Source Model": "Исходная модель",
    "System Prompt": "Системный промпт",
    "Text Encoder Checkpoints": "Чекпоинты текстового энкодера",
    "Use Current Model Settings as Default Settings": "Сделать настройки текущей модели настройками по умолчанию",
    "Max Objects": "Макс. объектов",
    "Max Time (s)": "Макс. время (сек)",
    "Negative Mask": "Инвертированная маска",
    "person, car, sky": "человек, машина, небо",
    "(recommended to keep it at 97)": "(рекомендуется оставить 97)",
    "Preview not yet Available": "Предпросмотр пока недоступен",
    "AVAILABLE": "ДОСТУПНО",
    "Adaptive Projected Guidance (requires Guidance > 1 or Audio Guidance > 1)": "Адаптивное проецируемое наведение (нужен Guidance > 1 или Audio Guidance > 1)",
    "Advanced Mode": "Продвинутый режим",
    "Automatic Removal of Background behind People or Objects in Reference Images": "Автоматическое удаление фона за людьми/объектами на референсных изображениях",
    "CFG Zero below this Layer (Extra Process)": "CFG Zero ниже этого слоя (доп. обработка)",
    "Capped By": "Ограничено",
    "Category": "Категория",
    "Certainty Percentage Skip": "Пропуск по проценту уверенности",
    "Classifier-Free Guidance Star (requires Guidance > 1)": "Classifier-Free Guidance Star (нужен Guidance > 1)",
    "Control Image": "Контрольное изображение",
    "Control Image to be Inpainted": "Контрольное изображение для инпейнтинга",
    "Control Video": "Контрольное видео",
    "Control Video / Control Audio / Positioned Frames Temporal Alignment when any Video to continue": "Синхронизация по времени контрольного видео/аудио/позиционированных кадров при продолжении видео",
    "Custom Checkbbox": "Свой чекбокс",
    "Custom Dropdown": "Свой выпадающий список",
    "Custom Guide": "Свой гайд",
    "Denoising Steps % end": "Денойзинг: конец диапазона шагов (%)",
    "Denoising Steps % start": "Денойзинг: начало диапазона шагов (%)",
    "Denoising Strength (the Lower the Closer to the Control Image/Video)": "Сила денойзинга (чем меньше — тем ближе к контрольному изображению/видео)",
    "Early Stop": "Ранняя остановка",
    "Embedded Guidance Scale": "Встроенный масштаб наведения (Guidance Scale)",
    "End Image(s)": "Конечное(ые) изображение(я)",
    "Enhanced Prompt": "Улучшенный промпт",
    "Film Grain Intensity (0 = disabled)": "Интенсивность зернистости плёнки (0 = выкл.)",
    "Film Grain Saturation": "Насыщенность зернистости плёнки",
    "Frames to keep in Control Video (empty=All, 1=first, a:b for a range, space to separate values)": "Какие кадры оставить в контрольном видео (пусто=все, 1=первый, a:b — диапазон, пробел — разделитель)",
    "Generate additional frames before keeping the first image": "Сгенерировать дополнительные кадры перед сохранением первого изображения",
    "Generation References": "Референсы для генерации",
    "Guidance": "Наведение (Guidance)",
    "Guidance Phases": "Фазы наведения",
    "How to Process each Line of the Text Prompt": "Как обрабатывать каждую строку текстового промпта",
    "Image Mask Area (for Inpainting, white = Control Area, black = Unchanged)": "Область маски изображения (для инпейнтинга: белое = зона правки, чёрное = без изменений)",
    "Images as starting points for new Videos in the Generation Queue": "Изображения как стартовые кадры для новых видео в очереди генерации",
    "Images as ending points for new Videos in the Generation Queue": "Изображения как конечные кадры для новых видео в очереди генерации",
    "Import Videos / Images / Audio Files": "Импортировать видео / изображения / аудио",
    "Include Media": "Включить медиафайлы",
    "Iterations": "Итерации",
    "Location": "Расположение",
    "Masking Strength (the Lower the More Freedom for Unmasked Area)": "Сила маскирования (чем меньше — тем больше свободы вне маски)",
    "Expand / Shrink Mask Area": "Расширить / сузить область маски",
    "Max Duration": "Макс. длительность",
    "Media to Import in Galleries": "Медиафайлы для импорта в галереи",
    "Model Switch": "Переключение модели",
    "Multiple Images as Texts Prompts": "Несколько изображений как текстовые промпты",
    "Negative Prompt": "Негативный промпт",
    "Normalize Audio Volumes": "Нормализовать громкость аудио",
    "Ignore Background Music (for better LipSync)": "Игнорировать фоновую музыку (для лучшей синхронизации губ)",
    "Number of Images": "Количество изображений",
    "Number of Inference Steps": "Количество шагов генерации",
    "Number of frames": "Количество кадров",
    "Outer Box Resolution (one dimension may be less to preserve video W/H ratio)": "Разрешение внешней рамки (одна сторона может быть меньше для сохранения пропорций видео)",
    "Output Filename (Leave Blank for Auto Naming)": "Имя файла результата (оставь пустым для автоназвания)",
    "Output Resolution (Input Images wil be Cropped if the W/H ratio is different)": "Разрешение результата (входные изображения обрежутся, если пропорции не совпадают)",
    "Pause between Multi Speakers sentences (seconds)": "Пауза между репликами разных дикторов (сек)",
    "Perturbation": "Возмущение (Perturbation)",
    "Perturbation Layers": "Слои возмущения",
    "Phases": "Фазы",
    "Pose": "Поза",
    "Positions of Injected Frames (1=first, L=window end, X=skip window; no position for other Image Refs)": "Позиции внедрённых кадров (1=первый, L=конец окна, X=пропустить окно; для других референсов позиция не указывается)",
    "Preview": "Предпросмотр",
    "RIFLEx positional embedding to generate long video": "Позиционные эмбеддинги RIFLEx для генерации длинного видео",
    "Remove Background Music / Noise": "Убрать фоновую музыку / шум",
    "Remux Audio": "Ремукс аудио",
    "Rescale Internaly Image Ref (% in relation to Output Video) to change Output Composition": "Внутреннее масштабирование референсного изображения (% от результата), меняет композицию кадра",
    "Resolution Budget (Pixels will be reallocated to preserve Inputs W/H ratio)": "Бюджет разрешения (пиксели перераспределятся для сохранения пропорций входа)",
    "Sampler Solver / Scheduler": "Сэмплер / планировщик (Scheduler)",
    "Seed (-1 for random)": "Seed (-1 — случайное значение)",
    "Self Refiner": "Само-рефайнер",
    "Skip Steps Cache Type": "Тип кэша пропуска шагов",
    "Skip Steps starting moment in % of generation": "Момент начала пропуска шагов (% генерации)",
    "Speakers Locations separated by a Space. Each Location = Left:Right or a BBox Left:Top:Right:Bottom": "Расположение дикторов через пробел. Формат: Лево:Право или рамка Лево:Верх:Право:Низ",
    "Start - End": "Начало — конец",
    "Start / Reference Images": "Стартовые / референсные изображения",
    "Start Image": "Стартовое изображение",
    "Start-End": "Начало-конец",
    "Step Range": "Диапазон шагов",
    "Think": "Думать",
    "To Audio Source": "В источник аудио",
    "To Audio Source 2": "В источник аудио 2",
    "To Control Image": "В контрольное изображение",
    "To Control Video": "В контрольное видео",
    "To End Image": "В конечное изображение",
    "To Mask Image": "В маску изображения",
    "To Reference Image": "В референсное изображение",
    "To Soundtrack": "В саундтрек",
    "To Start Image": "В стартовое изображение",
    "To Video Source": "В источник видео",
    "Truncate Video beyond this number of resampled Frames (empty=Keep All, negative truncates from End)": "Обрезать видео после этого числа кадров (пусто=оставить все, отрицательное — с конца)",
    "Uncertainty Threshold": "Порог неопределённости",
    "Video Length": "Длина видео",
    "Video Mask": "Маска видео",
    "Video Mask Area (for Inpainting, white = Control Area, black = Unchanged)": "Область маски видео (для инпейнтинга: белое = зона правки, чёрное = без изменений)",
    "Voice to follow": "Голос для повторения",
    "Voice to follow #2": "Голос для повторения №2",
    "Enable Spatial Outpainting on Control Video, Landscape or Positioned Reference Frames": "Включить пространственный аутпейнтинг для контрольного видео, ландшафтных или позиционированных референсных кадров",
    "Top %": "Сверху %",
    "Bottom %": "Снизу %",
    "Left %": "Слева %",
    "Right %": "Справа %",
    "Reference Images": "Референсные изображения",
    "Input Video Strength": "Сила входного видео",
    "Audio Option": "Настройка аудио",
    "➕ Add": "➕ Добавить",
    "Enhance Prompt using a LLM": "Улучшить промпт через LLM",
    "Based on Text Prompt (No Images Selected)": "На основе текстового промпта (без изображений)",
    "Based on Text Prompt and Images": "На основе текста и изображений",
    "Based on Text Prompt Content": "На основе содержимого текстового промпта",
    "End Images": "Конечные изображения",
    "Injected Frames": "Внедрённые кадры",
    "Main Ref. Image": "Основное реф. изображение",
    "Ref. Images": "Реф. изображения",
    "Ref Image": "Реф. изображение",
    "Aligned to the beginning of the First Window of the new Video Sample": "Выровнено по началу первого окна нового видео",
    "Aligned to the beginning of the Source Video": "Выровнено по началу исходного видео",
    "Always OFF": "Всегда выкл.",
    "Always ON": "Всегда вкл.",
    "Applying Audio Post Processing": "Применяется постобработка звука…",
    "Applying Audio Remuxing": "Применяется ремукс аудио…",
    "Applying Media Post Processing": "Применяется постобработка медиа…",
    "Auto (ON if Video longer than 5s)": "Авто (вкл., если видео длиннее 5с)",
    "Auto fps: Control Video if any, or Model Default": "Авто fps: как у контрольного видео, иначе по умолчанию модели",
    "Auto fps: Source Video if any, or Control Video if any, or Model Default": "Авто fps: как у исходного видео, иначе как у контрольного, иначе по умолчанию модели",
    "Auto fps: Source Video if any, or Model Default": "Авто fps: как у исходного видео, иначе по умолчанию модели",
    "Auto: Best available (sage2 > sage > sdpa)": "Авто: лучший доступный (sage2 > sage > sdpa)",
    "Control Length": "Длина по контролю",
    "Control Video fps": "FPS контрольного видео",
    "Default Attention Mode": "Attention-режим по умолчанию",
    "Disabled": "Отключено",
    "Enabled with P1-Norm": "Включено с P1-Norm",
    "Enabled with P2-Norm": "Включено с P2-Norm",
    "First Block Cache": "Кэш первого блока",
    "Fit into a 16:9 Box": "Вписать в рамку 16:9",
    "Fit into a 1:1 Box": "Вписать в рамку 1:1",
    "Fit into a 21:9 Box": "Вписать в рамку 21:9",
    "Fit into a 3:4 Box": "Вписать в рамку 3:4",
    "Fit into a 4:3 Box": "Вписать в рамку 4:3",
    "Fit into a 9:16 Box": "Вписать в рамку 9:16",
    "Fit into a 9:21 Box": "Вписать в рамку 9:21",
    "Generate every combination of images and texts": "Сгенерировать все комбинации изображений и текстов",
    "Generating...": "Генерация…",
    "Keep Backgrounds behind all Reference Images": "Сохранить фон за всеми референсными изображениями",
    "Manual Expansion": "Ручное расширение",
    "Match images and text prompts": "Сопоставить изображения и текстовые промпты",
    "None": "Нет",
    "Nothing": "Ничего",
    "OFF": "Выкл.",
    "ON": "Вкл.",
    "One Phase": "Одна фаза",
    "Two Phases": "Две фазы",
    "Two Phases with Tiling": "Две фазы с тайлингом",
    "Three Phases": "Три фазы",
    "People / Objects": "Люди / объекты",
    "Phase 1-2 transition": "Переход фаза 1→2",
    "Phase 2-3 transition": "Переход фаза 2→3",
    "Please enter a name for a Lora Preset / Settings file": "Введите название пресета LoRA / файла настроек",
    "Profile 1, HighRAM_HighVRAM: at least 64 GB of RAM and 24 GB of VRAM, the fastest for short videos with a RTX 3090 / RTX 4090": "Профиль 1, HighRAM_HighVRAM: минимум 64 ГБ RAM и 24 ГБ VRAM — самый быстрый для коротких видео на RTX 3090 / RTX 4090",
    "Profile 3+, VeryLowRAM_HighVRAM: at least 32 GB of RAM and 24 GB of VRAM, variant of Profile 3 that won't used Reserved Memory to reduce RAM usage": "Профиль 3+, VeryLowRAM_HighVRAM: минимум 32 ГБ RAM и 24 ГБ VRAM — вариант профиля 3 без резервируемой памяти для снижения расхода RAM",
    "Profile 3, LowRAM_HighVRAM: at least 32 GB of RAM and 24 GB of VRAM, adapted for RTX 3090 / RTX 4090 with limited RAM for good speed short video": "Профиль 3, LowRAM_HighVRAM: минимум 32 ГБ RAM и 24 ГБ VRAM — для RTX 3090/4090 с ограниченной RAM, хорошая скорость на коротких видео",
    "Profile 4+, LowRAM_LowVRAM+: at least 32 GB of RAM and 12 GB of VRAM, variant of Profile 4, slightly slower but needs less VRAM": "Профиль 4+, LowRAM_LowVRAM+: минимум 32 ГБ RAM и 12 ГБ VRAM — вариант профиля 4, чуть медленнее, но нужно меньше VRAM",
    "Profile 4, LowRAM_LowVRAM (Recommended): at least 32 GB of RAM and 12 GB of VRAM, if you have little VRAM or want to generate longer videos": "Профиль 4, LowRAM_LowVRAM (рекомендуется): минимум 32 ГБ RAM и 12 ГБ VRAM — если мало VRAM или нужны более длинные видео",
    "Profile 5, VerylowRAM_LowVRAM (Fail safe): at least 24 GB of RAM and 10 GB of VRAM, if you don't have much it won't be fast but maybe it will work": "Профиль 5, VerylowRAM_LowVRAM (аварийный): минимум 24 ГБ RAM и 10 ГБ VRAM — если совсем мало ресурсов, будет медленно, но может заработать",
    "Save All the Settings except Media": "Сохранить все настройки, кроме медиа",
    "Save All the Settings including Media": "Сохранить все настройки, включая медиа",
    "Save Loras & Only Prompt Comments": "Сохранить LoRA и только комментарии промпта",
    "Save Only Loras & Full Prompt": "Сохранить только LoRA и полный промпт",
    "Shortest generation, keep only the First Frame": "Кратчайшая генерация, оставить только первый кадр",
    "Skip Layer Guidance": "Пропуск наведения по слоям (Skip Layer Guidance)",
    "Source Video fps": "FPS исходного видео",
    "Start": "Начало",
    "around x1.5 speed up": "ускорение примерно x1.5",
    "around x1.75 speed up": "ускорение примерно x1.75",
    "around x2 speed up": "ускорение примерно x2",
    "around x2.25 speed up": "ускорение примерно x2.25",
    "around x2.5 speed up": "ускорение примерно x2.5",
    "sdpa: Default, always available": "sdpa: по умолчанию, всегда доступен",
    "start image": "стартовое изображение",
    "Top x": "Сверху x",
    "Bottom x": "Снизу x",
    "Left x": "Слева x",
    "Right x": "Справа x",
    "Num. of Generated Audio Files per Prompt": "Кол-во аудиофайлов на промпт",
    "Num. of Generated Videos per Prompt": "Кол-во видео на промпт",
    "Override Memory Profile": "Переопределить профиль памяти",
    "Override Attention Mode": "Переопределить режим внимания (Attention Mode)",
    "Video": "Видео",
    "Prompts (all the Lines are Parts of the Same Prompt, # lines = comments, ! lines = macros)": "Промпты (все строки — части одного промпта, строки с # — комментарии, строки с ! — макросы)",
    "Prompts (all the Lines are Parts of the Same Prompt, # lines = comments)": "Промпты (все строки — части одного промпта, строки с # — комментарии)",
    "Prompts (each Paragraph of Prompt separated by an Empty Line will be used for a Sliding Window, # lines = comments, ! lines = macros)": "Промпты (каждый абзац промпта, разделённый пустой строкой, будет использован для скользящего окна (Sliding Window), строки с # — комментарии, строки с ! — макросы)",
    "Prompts (each Paragraph of Prompt separated by an Empty Line will be used for a Sliding Window, # lines = comments)": "Промпты (каждый абзац промпта, разделённый пустой строкой, будет использован для скользящего окна (Sliding Window), строки с # — комментарии)",
    "Prompts (each Line of Prompt will be used for a Sliding Window, # lines = comments, ! lines = macros)": "Промпты (каждая строка промпта будет использована для скользящего окна (Sliding Window), строки с # — комментарии, строки с ! — макросы)",
    "Prompts (each Line of Prompt will be used for a Sliding Window, # lines = comments)": "Промпты (каждая строка промпта будет использована для скользящего окна (Sliding Window), строки с # — комментарии)",
    "Prompts (each Paragraph of Prompt separated by an Empty Line will generate a new Image, # lines = comments, ! lines = macros)": "Промпты (каждый абзац промпта, разделённый пустой строкой, создаст новое изображение, строки с # — комментарии, строки с ! — макросы)",
    "Prompts (each Paragraph of Prompt separated by an Empty Line will generate a new Image, # lines = comments)": "Промпты (каждый абзац промпта, разделённый пустой строкой, создаст новое изображение, строки с # — комментарии)",
    "Prompts (each Line of Prompt will generate a new Image, # lines = comments, ! lines = macros)": "Промпты (каждая строка промпта создаст новое изображение, строки с # — комментарии, строки с ! — макросы)",
    "Prompts (each Line of Prompt will generate a new Image, # lines = comments)": "Промпты (каждая строка промпта создаст новое изображение, строки с # — комментарии)",
    "Prompts (each Paragraph of Prompt separated by an Empty Line will generate a new Audio File, # lines = comments, ! lines = macros)": "Промпты (каждый абзац промпта, разделённый пустой строкой, создаст новое аудио, строки с # — комментарии, строки с ! — макросы)",
    "Prompts (each Paragraph of Prompt separated by an Empty Line will generate a new Audio File, # lines = comments)": "Промпты (каждый абзац промпта, разделённый пустой строкой, создаст новое аудио, строки с # — комментарии)",
    "Prompts (each Line of Prompt will generate a new Audio File, # lines = comments, ! lines = macros)": "Промпты (каждая строка промпта создаст новое аудио, строки с # — комментарии, строки с ! — макросы)",
    "Prompts (each Line of Prompt will generate a new Audio File, # lines = comments)": "Промпты (каждая строка промпта создаст новое аудио, строки с # — комментарии)",
    "Prompts (each Paragraph of Prompt separated by an Empty Line will generate a new Video, # lines = comments, ! lines = macros)": "Промпты (каждый абзац промпта, разделённый пустой строкой, создаст новое видео, строки с # — комментарии, строки с ! — макросы)",
    "Prompts (each Paragraph of Prompt separated by an Empty Line will generate a new Video, # lines = comments)": "Промпты (каждый абзац промпта, разделённый пустой строкой, создаст новое видео, строки с # — комментарии)",
    "Prompts (each Line of Prompt will generate a new Video, # lines = comments, ! lines = macros)": "Промпты (каждая строка промпта создаст новое видео, строки с # — комментарии, строки с ! — макросы)",
    "Prompts (each Line of Prompt will generate a new Video, # lines = comments)": "Промпты (каждая строка промпта создаст новое видео, строки с # — комментарии)",
  };

  // Подписи с "живыми" цифрами (длительность, fps) — точным совпадением не поймать,
  // переводим по regex, сохраняя захваченные числа как есть.
  var REGEX_MAP = [
    {
      re: /^Number of frames \((\d+) frames = 1s\), current duration: ([\d.]+)s$/,
      to: function (m) {
        return "Количество кадров (" + m[1] + " кадров = 1с), текущая длительность: " + m[2] + "с";
      },
    },
    {
      re: /^Override Frames Per Second \(model default=(\d+) fps\)$/,
      to: function (m) {
        return "Переопределить FPS (по умолчанию модели: " + m[1] + " fps)";
      },
    },
  ];

  // Фильтр семейств моделей рендерится как HTML-иконки: видимого текста нет,
  // текст живёт только в title/aria-label. Ключ — стабильный data-model-output-filter.
  var FAMILY_FILTER_TITLES = {
    all: "Все семейства моделей — показать полный список без фильтра.",
    video: "Видео-модели — генерация видео (текст в видео, изображение в видео и т.д.), включая Wan2.1/2.2.",
    image: "Модели изображений — создают только картинки, без видео.",
    audio: "Аудио-модели — генерация и обработка звука/озвучки, без видео.",
  };

  // Иконки-инструменты рядом с выбором модели (⌕ + ↻ ⏏) — тоже стабильные elem_id.
  var TOOL_TITLES = {
    wangp_model_tool_search: "Поиск моделей по названию",
    wangp_model_tool_finetune: "Создать свою модель (файнтюн) на основе текущей",
    wangp_model_tool_refresh: "Обновить список моделей",
    wangp_model_tool_unload: "Выгрузить модель из памяти GPU",
  };

  var GUIDE_HTML =
    '<b>Как выбрать модель:</b> слева — семейство (например, Wan2.1/2.2), справа — конкретная версия ' +
    "(размер вроде 14B/5B, тип T2V/I2V и т.д.). Чем больше цифра — тем выше качество, но дольше генерация " +
    "и больше нужно видеопамяти. Наведи курсор на цветные иконки слева, чтобы увидеть подсказки по фильтру.";

  var WORKFLOW_GUIDE_HTML =
    "<b>Как сделать видео — выбери один из режимов ниже:</b><br>" +
    "• <b>Из текста:</b> оставь «Новое видео», убедись что открыта вкладка «Текст в видео», впиши описание в поле промпта ниже и нажми «Сгенерировать».<br>" +
    "• <b>Из картинки:</b> выбери «Начать с изображения» (появляется, если модель это умеет) и загрузи картинку — из неё оживёт видео.<br>" +
    "• <b>Продолжить готовое видео:</b> «Продолжить видео» — загрузить своё видео и продлить его; «Продолжить последнее видео» — продолжить то, что уже сгенерировано здесь.";

  function translateFamilyFilter(root) {
    var buttons = root.querySelectorAll("#wangp_model_output_filter button[data-model-output-filter]");
    for (var i = 0; i < buttons.length; i++) {
      var btn = buttons[i];
      var key = btn.getAttribute("data-model-output-filter");
      var ruTitle = FAMILY_FILTER_TITLES[key];
      if (ruTitle) {
        btn.title = ruTitle;
        btn.setAttribute("aria-label", ruTitle);
      }
    }
  }

  function translateToolIcons(root) {
    for (var id in TOOL_TITLES) {
      var el = root.getElementById ? root.getElementById(id) : document.getElementById(id);
      if (el) {
        el.title = TOOL_TITLES[id];
        el.setAttribute("aria-label", TOOL_TITLES[id]);
      }
    }
  }

  function translateTabs(root) {
    var tabs = root.querySelectorAll("[data-tab-id]");
    for (var i = 0; i < tabs.length; i++) {
      var el = tabs[i];
      var id = el.getAttribute("data-tab-id");
      var ruLabel = TAB_LABELS[id];
      if (!ruLabel) continue;
      if (!el.dataset.ruOriginal) el.dataset.ruOriginal = el.textContent.trim();
      if (el.textContent.trim() === el.dataset.ruOriginal && el.dataset.ruOriginal !== ruLabel) {
        el.textContent = ruLabel;
      }
    }
  }

  function translateText(root) {
    var walker = document.createTreeWalker(root, NodeFilter.SHOW_TEXT, null);
    var node;
    var nodes = [];
    while ((node = walker.nextNode())) nodes.push(node);
    for (var i = 0; i < nodes.length; i++) {
      var tn = nodes[i];
      var trimmed = tn.nodeValue.trim();
      if (!trimmed) continue;
      if (TEXT_MAP.hasOwnProperty(trimmed)) {
        tn.nodeValue = tn.nodeValue.replace(trimmed, TEXT_MAP[trimmed]);
        continue;
      }
      for (var r = 0; r < REGEX_MAP.length; r++) {
        var rule = REGEX_MAP[r];
        var match = trimmed.match(rule.re);
        if (match) {
          tn.nodeValue = tn.nodeValue.replace(trimmed, rule.to(match));
          break;
        }
      }
    }
  }

  function insertGuide(root) {
    if (root.querySelector(".ru-model-guide")) return;
    var filterBlock = document.getElementById("wangp_model_output_filter");
    if (!filterBlock) return;
    var rowAncestor = filterBlock.closest(".row") || filterBlock.parentElement;
    if (!rowAncestor || !rowAncestor.parentElement) return;
    var box = document.createElement("div");
    box.className = "ru-model-guide";
    box.innerHTML = GUIDE_HTML;
    rowAncestor.parentElement.insertBefore(box, rowAncestor.nextSibling);
  }

  function insertWorkflowGuide(root) {
    var labels = root.querySelectorAll('[data-testid="New Video-radio-label"]');
    for (var i = 0; i < labels.length; i++) {
      var block = labels[i].closest(".block");
      if (!block || !block.parentElement) continue;
      if (block.parentElement.querySelector(":scope > .ru-workflow-guide")) continue;
      var box = document.createElement("div");
      box.className = "ru-workflow-guide";
      box.innerHTML = WORKFLOW_GUIDE_HTML;
      block.parentElement.insertBefore(box, block.nextSibling);
    }
  }

  function runPass() {
    translateTabs(document.body);
    translateText(document.body);
    translateFamilyFilter(document.body);
    translateToolIcons(document.body);
    insertGuide(document.body);
    insertWorkflowGuide(document.body);
  }

  var debounceTimer = null;
  var observer = new MutationObserver(function () {
    clearTimeout(debounceTimer);
    debounceTimer = setTimeout(runPass, 150);
  });
  observer.observe(document.body, { childList: true, subtree: true, characterData: true });

  var attempts = 0;
  var bootstrap = setInterval(function () {
    runPass();
    attempts += 1;
    if (attempts > 20) clearInterval(bootstrap);
  }, 500);
})();
/* === RU-UI END === */
'''

css_path = WAN2GP_ROOT / 'shared' / 'gradio' / 'ui_styles.css'
js_path = WAN2GP_ROOT / 'shared' / 'gradio' / 'ui_scripts.js'

for path, payload in ((css_path, RU_CSS), (js_path, RU_JS)):
    current = path.read_text(encoding='utf-8') if path.exists() else ''
    marker = '=== RU-UI START ==='
    if marker in current:
        before, _, after_marker = current.partition(marker)
        rest = after_marker.split('=== RU-UI END ===', 1)
        tail = rest[1] if len(rest) > 1 else ''
        current = before.rstrip() + '\n' + payload.strip() + '\n' + tail.lstrip()
        path.write_text(current, encoding='utf-8')
        print(f'Обновили русский слой в {path.name}')
    else:
        with path.open('a', encoding='utf-8') as f:
            f.write('\n' + payload)
        print(f'Добавили русский слой в {path.name}')

print('Готово. Если Wan2GP уже запущен — перезапусти Шаг 6, чтобы подтянуть изменения.')

## Шаг 6. Запускаем Wan2GP

Запускаем веб-интерфейс Gradio. Ссылка появится в выводе этой ячейки — перейди по ней, чтобы открыть интерфейс в браузере. Чтобы не потерять соединение, не останавливай ячейку; для завершения работы нажми квадратную кнопку **Stop**.

По умолчанию интерфейс запускается с **закреплённой рекомендуемой моделью** (Wan2.1 Text-to-Video 14B) — весь выбор семейств/версий/фильтров моделей скрыт, остаётся только поле промпта и кнопка "Сгенерировать". Если хочешь сам выбирать модель — поставь `LOCK_TO_RECOMMENDED_MODEL = False` в ячейке ниже.

In [ ]:
import os, subprocess, sys, threading, time

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')

# LOCK_TO_RECOMMENDED_MODEL=True прячет весь выбор модели (семейство/версия/фильтры) —
# остаётся только название модели, поле промпта и кнопка "Сгенерировать".
# Поставь False, если хочешь сам переключаться между моделями Wan2GP.
LOCK_TO_RECOMMENDED_MODEL = True

cmd = [
    sys.executable,
    '-u',
    'wgp.py',
    '--listen',
    '--server-port', '7860',
    '--share',
    '--profile', '5',
]
if LOCK_TO_RECOMMENDED_MODEL:
    cmd += ['--t2v', '--lock-model']

if USE_GOOGLE_DRIVE_DATA:
    print('Используем Google Drive для чекпоинтов, LoRA, результатов и кэша.')
else:
    print('Используем диск сессии Colab для чекпоинтов, LoRA, результатов и кэша.')
if LOCK_TO_RECOMMENDED_MODEL:
    print('Модель закреплена: Wan2.1 Text-to-Video 14B (рекомендуемая по умолчанию). Выбор моделей скрыт.')
print('Запускаем Wan2GP…')
process = subprocess.Popen(
    cmd,
    cwd=str(WAN2GP_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
stop_event = threading.Event()

def keepalive():
    while not stop_event.is_set():
        time.sleep(45)
        if stop_event.is_set():
            break
        print('[keepalive] Ячейка всё ещё работает…')

keepalive_thread = threading.Thread(target=keepalive, daemon=True)
keepalive_thread.start()

try:
    for line in iter(process.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Останавливаем Wan2GP…')
    process.terminate()
finally:
    stop_event.set()
    process.wait()
    keepalive_thread.join(timeout=1)
    print(f'Wan2GP остановлен (код возврата: {process.returncode}).')